# CONFIG

In [ ]:
import argparse
import os
import pandas as pd
import numpy as np

from dataclasses import dataclass
from maikol_utils.print_utils import print_separator


pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 500)

%load_ext autoreload
%autoreload 2
%matplotlib inline

In [ ]:
@dataclass
class Configuration:
    """Configuration dataclass to hold application settings."""
    DATA_PATH: str = "../data/scores"
    
    data_name: str = "A"
    clients_path: str = ""
    impostors_path: str = ""

    # ========================================
    threshold: float = 0.1
    

    def __post_init__(self):
        self.clients_path = os.path.join(self.DATA_PATH, f"scores{self.data_name}_clientes")
        self.impostors_path = os.path.join(self.DATA_PATH, f"scores{self.data_name}_impostores")


# CODE

## Load data

In [ ]:
CONFIG_A = Configuration(data_name="A")
CONFIG_B = Configuration(data_name="B")

print(f"Data A at {CONFIG_A.clients_path} & {CONFIG_A.impostors_path}")
print(f"Data B at {CONFIG_B.clients_path} & {CONFIG_B.impostors_path}")

# Load A
df_clients_A = pd.read_csv(CONFIG_A.clients_path, sep=" ", header=None, names=["id", "score"])
df_impostors_A = pd.read_csv(CONFIG_A.impostors_path, sep=" ", header=None, names=["id", "score"])
df_clients_A["client"] = True
df_impostors_A["client"] = False
df_all_A = pd.concat([df_clients_A, df_impostors_A])

# Load B
df_clients_B = pd.read_csv(CONFIG_B.clients_path, sep=" ", header=None, names=["id", "score"])
df_impostors_B = pd.read_csv(CONFIG_B.impostors_path, sep=" ", header=None, names=["id", "score"])
df_clients_B["client"] = True
df_impostors_B["client"] = False
df_all_B = pd.concat([df_clients_B, df_impostors_B])

print("\nDataset A sample:")
print(df_all_A.sample(10))
print("\nDataset B sample:")
print(df_all_B.sample(10))


In [ ]:
print("Dataset A:")
print(df_all_A.describe())
print("\nDataset B:")
print(df_all_B.describe())


## Compute

In [ ]:
def compute_FP(df, threshold):
    impostors = df[~df["client"]]
    return impostors[impostors["score"] > threshold].shape[0] / impostors.shape[0]

def compute_FN(df, threshold):
    clients = df[df["client"]]
    return clients[clients["score"] <= threshold].shape[0] / clients.shape[0]


threshold = 0.1 # Example
print(f"Dataset A - FP: {compute_FP(df_all_A, threshold):.4f}, FN: {compute_FN(df_all_A, threshold):.4f}")
print(f"Dataset B - FP: {compute_FP(df_all_B, threshold):.4f}, FN: {compute_FN(df_all_B, threshold):.4f}")


In [ ]:
def get_thresholds(df):
    scores = df["score"].unique().tolist()
    return sorted(scores + [
        df["score"].min() - 1e-6,  # below all scores → FPR=1, FNR=0 → point (1, 0)
        df["score"].max() + 1e-6,  # above all scores → FPR=0, FNR=1 → point (0, 1)
    ])

thresholds_A = get_thresholds(df_all_A)
thresholds_B = get_thresholds(df_all_B)
# thresholds_A = sorted(df_all_A["score"].unique().tolist() + [0, 1])
# thresholds_B = sorted(df_all_B["score"].unique().tolist() + [0, 1])

# Dataset A
fps_A = {
    ths: compute_FP(df_all_A, ths)
    for ths in thresholds_A
}
fns_A = {
    ths: compute_FN(df_all_A, ths)
    for ths in thresholds_A
}

# Dataset B
fps_B = {
    ths: compute_FP(df_all_B, ths)
    for ths in thresholds_B
}
fns_B = {
    ths: compute_FN(df_all_B, ths)
    for ths in thresholds_B
}


In [ ]:
import plotly.graph_objects as go

fig = go.Figure()

# Dataset B
fig.add_trace(go.Scatter(
    x=list(fps_B.values()),
    y=[1 - fn for fn in fns_B.values()],  # TPR = 1 - FNR
    mode='lines+markers',
    name='Dataset B',
    marker=dict(
        size=2,
        color='orangered',
    ),
    line=dict(color='orangered', width=3),
    text=[f"Threshold: {ths:.4f}" for ths in thresholds_B],
    hovertemplate='<b>Dataset B</b><br><b>FPR:</b> %{x:.4f}<br><b>FNR:</b> %{y:.4f}<br>%{text}<extra></extra>'
))
# Dataset A
fig.add_trace(go.Scatter(
    x=list(fps_A.values()),
    y=[1 - fn for fn in fns_A.values()],  # TPR = 1 - FNR
    mode='lines+markers',
    name='Dataset A',
    marker=dict(
        size=2,
        color='royalblue',
    ),
    line=dict(color='royalblue', width=3),
    text=[f"Threshold: {ths:.4f}" for ths in thresholds_A],
    hovertemplate='<b>Dataset A</b><br><b>FPR:</b> %{x:.4f}<br><b>FNR:</b> %{y:.4f}<br>%{text}<extra></extra>'
))


fig.update_layout(
    title="ROC Curve - False Positive vs False Negative Rate (Datasets A & B)",
    xaxis_title="False Positive Rate",
    yaxis_title="False Negative Rate",
    xaxis=dict(range=[0, 1]),
    yaxis=dict(range=[0, 1]),
    hovermode='closest',
    template='plotly_white',
    width=800,
    height=700,
    font=dict(size=12),
    showlegend=True
)

fig.show()


### Area bajo la curva

In [ ]:
def compute_auc(fps, fns):
    roc_points = sorted(zip(fps, fns), key=lambda p: p[0])
    fps_sorted, fns_sorted = zip(*roc_points)

    auc = 0
    for i in range(1, len(fps_sorted)):
        fp, fn = fps_sorted[i], fns_sorted[i]
        fp_, fn_ = fps_sorted[i-1], fns_sorted[i-1]
        auc += (fn + fn_) * 0.5 * (fp - fp_)

    return 1 - auc  

auc_A = compute_auc(fps_A.values(), fns_A.values())
auc_B = compute_auc(fps_B.values(), fns_B.values())

print(f"AUC Dataset A: {auc_A:.4f}")
print(f"AUC Dataset B: {auc_B:.4f}")


### FP(FN=X) y umbral

In [ ]:
def compute_fp_given_fn_x(fns, fps, target_fn):
    dif = [abs(fn - target_fn) for fn in fns.values()]
    closest_fn = sorted(zip(dif, fns.items()), key=lambda p: p[0])[0]
    thrs = closest_fn[1][0]

    return {
        "thrs": thrs,
        "fn": closest_fn[1][1],
        "fp": fps[thrs],
    }

def format_metric_dict(metric_dict):
    return (
        f"{'{' }'thrs': {metric_dict['thrs']:.8f}, "
        f"'fn': {metric_dict['fn'] * 100:.2f}%, "
        f"'fp': {metric_dict['fp'] * 100:.2f}%{'}'}"
    )

for x in [0.01, 0.05, 0.2]:
    print(f"FP at FN={x} (Dataset A): {format_metric_dict(compute_fp_given_fn_x(fns_A, fps_A, x))}")
    print(f"FP at FN={x} (Dataset B): {format_metric_dict(compute_fp_given_fn_x(fns_B, fps_B, x))}")


### FN(FP=X) y umbral

In [ ]:
def compute_fn_given_fp_x(fns, fps, target_fp):
    dif = [abs(fp - target_fp) for fp in fps.values()]
    closest_fp = sorted(zip(dif, fps.items()), key=lambda p: p[0])[0]
    thrs = closest_fp[1][0]

    return {
        "thrs": thrs,
        "fn": fns[thrs],
        "fp": closest_fp[1][1],
    }

for x in [0.01, 0.05, 0.2]:
    print(f"FN at FP={x} (Dataset A): {format_metric_dict(compute_fn_given_fp_x(fns_A, fps_A, x))}")
    print(f"FN at FP={x} (Dataset B): {format_metric_dict(compute_fn_given_fp_x(fns_B, fps_B, x))}")


### FN = FP y umbral


In [ ]:
def compute_fn_eq_fp(fns, fps):
    # dif = [abs(fp - fn) for fn, fp in zip(fns.values(), fps.values())]
    dif = [abs(fps[t] - fns[t]) for t in fns]

    closest_fp = sorted(zip(dif, fps.items(), fns.items()), key=lambda p: p[0])[0]
    thrs = closest_fp[1][0]

    return {
        "thrs": thrs,
        "fn": closest_fp[2][1],
        "fp": closest_fp[1][1],
    }

print(f"FN = FP (Dataset A): {format_metric_dict(compute_fn_eq_fp(fns_A, fps_A))}")
print(f"FN = FP (Dataset B): {format_metric_dict(compute_fn_eq_fp(fns_B, fps_B))}")

### D'

In [13]:
def compute_dprime(df):
    pos = df[df["client"]]["score"]
    neg = df[~df["client"]]["score"]
    d = (pos.mean() - neg.mean()) / np.sqrt(pos.var() + neg.var())
    return d

print(f"D' (Dataset A): {compute_dprime(df_all_A):.4f}")
print(f"D' (Dataset B): {compute_dprime(df_all_B):.4f}")

D' (Dataset A): 0.7597
D' (Dataset B): 0.8732
